In [ ]:
%store -r X_small y indexes X

In [ ]:
import numpy as np
import pandas as pd
import glob
from sklearn.datasets import load_iris
from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
import copy
from sklearn.ensemble import RandomForestClassifier

In [ ]:
def load_data():
    path = f'./python'
    files = glob.glob(path + '/test*.npy')
    files = sorted(files)
    
    result = []
    labels = [file.split('/')[-1].replace('.npy', '') for file in files]
    
    for i, file in enumerate(files):
        print(f"Loaded {i + 1}/{len(files)} test files.", end="\r")
        data = np.load(file)
        result.append(data)
        
    result = np.array(result)
    
    return result, labels

In [ ]:
test_data, labels = load_data()

In [ ]:
sample = test_data[0]
labels

# Neural Network

In [ ]:
# class ComplexNet(nn.Module):
#     def __init__(self):
#         super(ComplexNet, self).__init__()
        
#         # CNN layers
#         self.conv1 = nn.Conv1d(1, 256, kernel_size=3, stride=1, padding=1)
#         self.bn1 = nn.BatchNorm1d(256)
#         self.conv2 = nn.Conv1d(256, 128, kernel_size=3, stride=1, padding=1)
#         self.bn2 = nn.BatchNorm1d(128)
        
#         # RNN layer
#         self.gru = nn.GRU(input_size=128, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
        
#         # Fully connected layers
#         self.fc1 = nn.Linear(64*2, 128)  # 2 for bidirectional outputs
#         self.dropout = nn.Dropout(0.2)
#         self.fc2 = nn.Linear(128, 7)
    
#     def forward(self, x):
#         # CNN layers
#         x = x.unsqueeze(1)  # (N, 548) -> (N, 1, 548)
#         x = F.relu(self.bn1(self.conv1(x)))
#         x = F.relu(self.bn2(self.conv2(x)))
        
#         # Global Average Pooling
#         x = torch.mean(x, 2)
        
#         # RNN layer
#         x = x.unsqueeze(1)  # (N, 128) -> (N, 1, 128) for RNN
#         _, h_n = self.gru(x)
#         x = torch.cat((h_n[-2,:,:], h_n[-1,:,:]), dim=1)  # Concat the last hidden state of layers
        
#         # Fully connected layers
#         x = F.relu(self.fc1(x))
#         x = self.dropout(x)
#         x = self.fc2(x)
        
#         return x

In [ ]:
class BirdClassifier(nn.Module):
    def __init__(self):
        super(BirdClassifier, self).__init__()
        
        self.fc1 = nn.Linear(548, 1024)
        self.bn1 = nn.BatchNorm1d(1024)
        
        self.fc2 = nn.Linear(1024, 512)
        self.bn2 = nn.BatchNorm1d(512)
        
        self.fc3 = nn.Linear(512, 256)
        self.bn3 = nn.BatchNorm1d(256)

        self.fc4 = nn.Linear(256, 128)
        self.bn4 = nn.BatchNorm1d(128)

        self.fc5 = nn.Linear(128, 64)
        self.bn5 = nn.BatchNorm1d(64)
        
        self.dropout = nn.Dropout(0.5)
        self.out = nn.Linear(64, 7)

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        x = F.relu(self.bn3(self.fc3(x)))
        x = F.relu(self.bn4(self.fc4(x)))
        x = F.relu(self.bn5(self.fc5(x)))
        x = self.dropout(x)
        x = self.out(x)
        return x


In [ ]:
# Define the loss function and optimizer
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Define the hyperparameters
learning_rate = 1e-5
batch_size = 128
num_epochs = 200

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert the data to PyTorch tensors
X_train = torch.from_numpy(X_train).float().to(device)
X_test = torch.from_numpy(X_test).float().to(device)
y_train = torch.from_numpy(y_train).long().to(device)
y_test = torch.from_numpy(y_test).long().to(device)

# Create the DataLoader for training and test sets
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

model = BirdClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Initialize variables for saving the best model
best_model_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0

# Train the model
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = running_corrects.to(torch.float32) / len(train_dataset)

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f}")

    # Evaluate the model on the test set
    model.eval()  # Set the model to evaluation mode
    running_loss = 0.0
    running_corrects = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(test_dataset)
    epoch_acc = running_corrects.to(torch.float32) / len(test_dataset)

    print(f"Epoch {epoch+1}/{num_epochs}, Test Loss: {epoch_loss:.4f}, Test Acc: {epoch_acc:.4f}")

    # Save the model if it has a higher accuracy than our current best
    if epoch_acc > best_acc:
        best_acc = epoch_acc
        best_model_wts = copy.deepcopy(model.state_dict())

# Load the best model weights
model.load_state_dict(best_model_wts)

In [ ]:
sub = ""

for i, sample in enumerate(test_data):
    y_test = sample[:, indexes]
    y_pred = clf.predict(y_test)
    y_pred = y_pred.astype(str)
    
    y_str = ','.join(y_pred)
    result = f'{labels[i]},{y_str}\n'
    sub += result
    
sub

In [ ]:
with open ('./submission_random_forest.csv', 'w') as file:
    file.write(sub)

# Neural Networks

# Simple Neural Network

In [ ]:
# Set the device (GPU if available, else CPU)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

# Define the neural network architecture
class BirdClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(BirdClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

class BirdClassifier(nn.Module):
    def __init__(self):
        super(BirdClassifier, self).__init__()
        
        self.fc1 = nn.Linear(50, 256)
        self.bn1 = nn.BatchNorm1d(256)
        
        self.fc2 = nn.Linear(256, 512)
        self.bn2 = nn.BatchNorm1d(512)
        
        self.fc3 = nn.Linear(512, 256)
        self.bn3 = nn.BatchNorm1d(256)

        self.fc4 = nn.Linear(256, 128)
        self.bn4 = nn.BatchNorm1d(128)

        self.fc5 = nn.Linear(128, 64)
        self.bn5 = nn.BatchNorm1d(64)
        
        self.dropout = nn.Dropout(0.5)
        self.out = nn.Linear(64, 7)

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        x = F.relu(self.bn3(self.fc3(x)))
        x = F.relu(self.bn4(self.fc4(x)))
        x = F.relu(self.bn5(self.fc5(x)))
        x = self.dropout(x)
        x = self.out(x)
        return x

# Define the hyperparameters
input_size = 50
hidden_size = 64
num_classes = 7
learning_rate = 1e-5
batch_size = 128
num_epochs = 200

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X_small, y, test_size=0.2, random_state=42)

# Convert the data to PyTorch tensors
X_train = torch.from_numpy(X_train).float().to(device)
X_test = torch.from_numpy(X_test).float().to(device)
y_train = torch.from_numpy(y_train).long().to(device)
y_test = torch.from_numpy(y_test).long().to(device)

# Create the DataLoader for training and test sets
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Create an instance of the BirdClassifier model
model = BirdClassifier().to(device)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Train the model
losses = []
accuracies = []

for epoch in range(num_epochs):
    model.train()
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Evaluate the model on the test set
    model.eval()
    with torch.no_grad():
        y_pred = []
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            y_pred.extend(predicted.cpu().numpy().tolist())
    
    acc = accuracy_score(y_test.cpu().numpy(), y_pred)
    losses.append(loss.item())
    accuracies.append(acc)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {acc:.4f}")

In [ ]:
class AudioNet(nn.Module):
    def __init__(self):
        super(AudioNet, self).__init__()
        self.conv1 = nn.Conv1d(1, 128, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(128*548, 256)
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, 7)

    def forward(self, x):
        x = x.view(x.shape[0], 1, -1)  # Reshape the input data
        x = F.relu(self.conv1(x))
        x = x.view(x.shape[0], -1)  # Flatten the data
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.log_softmax(self.fc3(x), dim=1)
        return x


In [ ]:
class AudioNet(nn.Module):
    def __init__(self):
        super(AudioNet, self).__init__()

        self.lstm = nn.LSTM(input_size=50, hidden_size=256, num_layers=2, batch_first=True, dropout=0.5)
        self.fc1 = nn.Linear(256, 64)
        self.fc2 = nn.Linear(64, 7)

    def forward(self, x):
        # LSTM layer expects input of shape (batch_size, seq_length, input_size)
        x = x.view(x.shape[0], 1, -1)  
        
        output, _ = self.lstm(x)
        
        # Use the output from the last time step
        output = output[:, -1, :]
        
        output = F.relu(self.fc1(output))
        output = F.log_softmax(self.fc2(output), dim=1)

        return output



In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

# Define the hyperparameters
learning_rate = 1e-5
batch_size = 128
num_epochs = 200

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X_small, y, test_size=0.2, random_state=42)

# Convert the data to PyTorch tensors
X_train = torch.from_numpy(X_train).float().to(device)
X_test = torch.from_numpy(X_test).float().to(device)
y_train = torch.from_numpy(y_train).long().to(device)
y_test = torch.from_numpy(y_test).long().to(device)

# Create the DataLoader for training and test sets
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Create an instance of the BirdClassifier model
model = AudioNet().to(device)

# Define the loss function and optimizer
criterion = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Train the model
losses = []
accuracies = []

for epoch in range(num_epochs):
    model.train()
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Evaluate the model on the test set
    model.eval()
    with torch.no_grad():
        y_pred = []
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            y_pred.extend(predicted.cpu().numpy().tolist())
    
    acc = accuracy_score(y_test.cpu().numpy(), y_pred)
    losses.append(loss.item())
    accuracies.append(acc)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {acc:.4f}")

# RNN

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

# Define the hyperparameters
learning_rate = 1e-5
batch_size = 32
num_epochs = 200

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X_small, y, test_size=0.2, random_state=42)

# Convert the data to PyTorch tensors
X_train = torch.from_numpy(X_train).float().to(device)
X_test = torch.from_numpy(X_test).float().to(device)
y_train = torch.from_numpy(y_train).long().to(device)
y_test = torch.from_numpy(y_test).long().to(device)

# Create the DataLoader for training and test sets
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
class AudioNetRNN(nn.Module):
    def __init__(self):
        super(AudioNetRNN, self).__init__()

        self.lstm = nn.LSTM(input_size=50, hidden_size=256, num_layers=2, batch_first=True, dropout=0.5)
        self.fc1 = nn.Linear(256, 64)
        self.fc2 = nn.Linear(64, 7)

    def forward(self, x):
        # LSTM layer expects input of shape (batch_size, seq_length, input_size)
        x = x.view(x.shape[0], 1, -1)  
        
        output, _ = self.lstm(x)
        
        # Use the output from the last time step
        output = output[:, -1, :]
        
        output = F.relu(self.fc1(output))
        output = F.log_softmax(self.fc2(output), dim=1)

        return output

In [ ]:
# Define the loss function and optimizer
model = AudioNetRNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Initialize variables for saving the best model
best_model_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0

# Train the model
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = running_corrects.to(torch.float32) / len(train_dataset)

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f}")

    # Evaluate the model on the test set
    model.eval()  # Set the model to evaluation mode
    running_loss = 0.0
    running_corrects = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(test_dataset)
    epoch_acc = running_corrects.to(torch.float32) / len(test_dataset)

    print(f"Epoch {epoch+1}/{num_epochs}, Test Loss: {epoch_loss:.4f}, Test Acc: {epoch_acc:.4f}")

    # Save the model if it has a higher accuracy than our current best
    if epoch_acc > best_acc:
        best_acc = epoch_acc
        best_model_wts = copy.deepcopy(model.state_dict())

# Load the best model weights
model.load_state_dict(best_model_wts)

In [ ]:
model.eval()

for i, sample in enumerate(test_data):
    y_test = sample[:, indexes].to(device)
    y_pred = model(y_test)
    print(y_pred)
#     y_pred = y_pred.astype(str)
    
#     y_str = ','.join(y_pred)
#     result = f'{labels[i]},{y_str}\n'
#     sub += result

In [ ]:
def predict(model, X_unlabeled):
    model.eval()  # Ensure the model is in evaluation mode
    predictions = []

    with torch.no_grad():
        for inputs in X_unlabeled:
            inputs = inputs.to(device)
            outputs = model(inputs.unsqueeze(0))  # Pass data through model, ensure the input tensor is (batch_size, features) shape
            _, predicted = torch.max(outputs, 1)  # Get the predicted classes
            predictions.extend(predicted.cpu().numpy().tolist())  # Add the predictions to our list

    return predictions

# Assuming X_unlabeled is a numpy array
X_unlabeled = torch.from_numpy(test_data[:, :, indexes]).float().to(device)

predictions = predict(model, X_unlabeled)


In [ ]:
test_data[:, :, indexes].shape

In [ ]:
# Assuming X_unlabeled is a numpy array
X_unlabeled = torch.from_numpy(test_data[:, :, indexes]).float().to(device)

# Create a DataLoader for the unlabeled data
unlabeled_dataset = TensorDataset(X_unlabeled)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=batch_size, shuffle=False)


# Prediction function
def predict(model, unlabeled_loader):
    model.eval()  # Ensure the model is in evaluation mode
    result = []
    predictions = []

    with torch.no_grad():
        for sample in test_data[:, :, indexes]:
            X_unlabeled = torch.from_numpy(sample).float().to(device)
            # Create a DataLoader for the unlabeled data
            unlabeled_dataset = TensorDataset(X_unlabeled)
            unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=batch_size, shuffle=False)
            
            for inputs in unlabeled_loader:
                inputs = inputs[0].to(device)
                outputs = model(inputs)  # Pass data through model
                _, predicted = torch.max(outputs, 1)  # Get the predicted classes
                predictions.extend(predicted.cpu().numpy().tolist())  # Add the predictions to our list
            
            result.append(predictions)
            predictions = []

    return np.array(result)

# Predict
predictions = predict(model, unlabeled_loader)


In [ ]:
sub = ""

for i, prediction in enumerate(predictions):
    y_pred = prediction.astype(str)
    
    y_str = ','.join(y_pred)
    result = f'{labels[i]},{y_str}\n'
    sub += result

In [ ]:
sub

In [ ]:
# with open ('./submission_neural_network.csv', 'w') as file:
#     file.write(sub)

In [ ]:
indexes